# Settings Modal

> Modal-based card stack settings with card count slider (auto toggle), width slider, and optional scale slider.

In [ ]:
#| default_exp components.settings_modal

In [ ]:
#| export
from typing import Any

from fasthtml.common import Div, Span, Input, Dialog, Button, Form, H3, Label, FT

from cjm_fasthtml_daisyui.components.actions.modal import modal, modal_box, modal_backdrop
from cjm_fasthtml_daisyui.components.actions.button import btn_modifiers
from cjm_fasthtml_daisyui.components.data_input.range_slider import range_dui, range_sizes
from cjm_fasthtml_daisyui.components.data_input.toggle import toggle, toggle_sizes
from cjm_fasthtml_daisyui.utilities.semantic_colors import text_dui, border_dui

from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import w, max_w
from cjm_fasthtml_tailwind.utilities.typography import font_size, font_weight
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, items, gap, justify, flex_direction,
)
from cjm_fasthtml_tailwind.utilities.layout import position, right, top
from cjm_fasthtml_tailwind.utilities.borders import border
from cjm_fasthtml_tailwind.core.base import combine_classes

from cjm_fasthtml_lucide_icons.factory import lucide_icon

# Design system recipes (V1 button roles, V11 icon-size roles)
from cjm_fasthtml_design_system.buttons import buttons
from cjm_fasthtml_design_system.icons import icons, IconSize

from cjm_fasthtml_card_stack.core.config import CardStackConfig
from cjm_fasthtml_card_stack.core.html_ids import CardStackHtmlIds

## Slider Sections

In [ ]:
#| export
def _render_section_label(
    text: str,  # label text
) -> Span:      # styled label element
    """Render a settings section label."""
    return Span(
        text,
        cls=combine_classes(font_size.sm, font_weight.medium, text_dui.base_content),
    )

In [ ]:
#| export
def _render_slider_with_labels(
    slider_id: str,   # HTML ID for the range input
    min_val: int,      # slider minimum
    max_val: int,      # slider maximum
    step_val: int,     # slider step
    current_val: int,  # current value
    low_label: str,    # label for low end
    high_label: str,   # label for high end
    oninput: str,      # JS oninput handler
    disabled: bool = False,  # whether slider is disabled
) -> Div:              # slider row with end labels
    """Render a range slider with low/high labels."""
    label_cls = combine_classes(font_size.xs, text_dui.base_content.opacity(50))

    attrs = dict(
        type="range",
        id=slider_id,
        min=str(min_val),
        max=str(max_val),
        step=str(step_val),
        value=str(current_val),
        cls=combine_classes(range_dui, range_sizes.sm, w.full),
        oninput=oninput,
    )
    if disabled:
        attrs['disabled'] = 'disabled'

    return Div(
        Div(
            Span(low_label, cls=label_cls),
            Span(high_label, cls=label_cls),
            cls=combine_classes(flex_display, justify.between, p.x(0.5)),
        ),
        Input(**attrs),
        cls=w.full,
    )

In [ ]:
from fasthtml.common import to_xml

html = to_xml(_render_slider_with_labels(
    slider_id="test-slider", min_val=30, max_val=120, step_val=5,
    current_val=80, low_label="Narrow", high_label="Wide",
    oninput="doStuff(this.value)",
))
assert 'id="test-slider"' in html
assert 'min="30"' in html
assert 'max="120"' in html
assert 'value="80"' in html
assert 'Narrow' in html
assert 'Wide' in html
assert 'disabled' not in html

# Test disabled
html_disabled = to_xml(_render_slider_with_labels(
    slider_id="test-slider", min_val=1, max_val=9, step_val=2,
    current_val=5, low_label="1", high_label="9",
    oninput="x()", disabled=True,
))
assert 'disabled' in html_disabled
print("Slider with labels tests passed!")

Slider with labels tests passed!


## Card Count Section

Card count uses an "Auto" toggle alongside a discrete range slider.
When auto mode is on, the slider is disabled and the card stack
auto-fits to the available viewport height.

In [ ]:
#| export
def _render_card_count_section(
    config: CardStackConfig,   # card stack config with visible_count_options
    ids: CardStackHtmlIds,     # HTML IDs for this instance
    current_count: int = 1,    # currently selected card count
    is_auto_mode: bool = True, # whether auto-adjust mode is active
) -> Div:                      # card count section with auto toggle and slider
    """Render the card count section with auto toggle and discrete slider."""
    opts = config.visible_count_options
    min_count = opts[0]
    max_count = opts[-1]
    step_count = opts[1] - opts[0] if len(opts) > 1 else 1
    prefix = config.prefix

    # JS: slider oninput calls handleCountChange with the numeric value
    slider_oninput = f"window.cardStacks['{prefix}'].handleCountChange(this.value)"

    # JS: toggle onchange — when checked (auto), call handleCountChange('auto') and disable slider;
    # when unchecked, call handleCountChange with slider value and enable slider
    toggle_onchange = (
        f"var slider=document.getElementById('{ids.card_count_slider}');"
        f"if(this.checked){{window.cardStacks['{prefix}'].handleCountChange('auto');slider.disabled=true;}}"
        f"else{{slider.disabled=false;window.cardStacks['{prefix}'].handleCountChange(slider.value);}}"
    )

    # Tick marks for discrete values
    tick_marks = Div(
        *[Span("|") for _ in opts],
        cls=combine_classes(justify.between, flex_display, p.x(0.5), font_size.xs,
                           text_dui.base_content.opacity(30)),
    )
    tick_labels = Div(
        *[Span(str(v)) for v in opts],
        cls=combine_classes(justify.between, flex_display, p.x(0.5), font_size.xs,
                           text_dui.base_content.opacity(50)),
    )

    slider_attrs = dict(
        type="range",
        id=ids.card_count_slider,
        min=str(min_count),
        max=str(max_count),
        step=str(step_count),
        value=str(current_count if not is_auto_mode else min_count),
        cls=combine_classes(range_dui, range_sizes.sm, w.full),
        oninput=slider_oninput,
    )
    if is_auto_mode:
        slider_attrs['disabled'] = 'disabled'

    return Div(
        # Label row: "Cards" on left, "Auto" toggle on right
        Div(
            _render_section_label("Cards"),
            Label(
                Span("Auto", cls=combine_classes(font_size.xs, text_dui.base_content.opacity(70))),
                Input(
                    type="checkbox",
                    id=ids.card_count_auto_toggle,
                    cls=combine_classes(toggle, toggle_sizes.sm),
                    checked=is_auto_mode,
                    onchange=toggle_onchange,
                ),
                cls=combine_classes(flex_display, items.center, gap(2)),
            ),
            cls=combine_classes(flex_display, items.center, justify.between, m.b(1)),
        ),
        # Slider
        Input(**slider_attrs),
        # Tick marks and labels
        tick_marks,
        tick_labels,
        cls=w.full,
    )

In [ ]:
from cjm_fasthtml_card_stack.core.config import CardStackConfig, _reset_prefix_counter
from cjm_fasthtml_card_stack.core.html_ids import CardStackHtmlIds

_reset_prefix_counter()
config = CardStackConfig(prefix="test")
ids = CardStackHtmlIds(prefix="test")

# Test auto mode (default)
section = _render_card_count_section(config, ids)
html = to_xml(section)
assert 'id="test-card-count-slider"' in html
assert 'id="test-card-count-auto-toggle"' in html
assert 'disabled="disabled"' in html  # slider disabled in auto mode
assert 'checked' in html  # toggle checked in auto mode
assert 'Auto' in html
assert 'Cards' in html
# Tick labels for default options (1, 3, 5, 7, 9)
for v in (1, 3, 5, 7, 9):
    assert f'>{v}<' in html
print("Card count section auto mode tests passed!")

# Test manual mode
section_manual = _render_card_count_section(config, ids, current_count=5, is_auto_mode=False)
html_manual = to_xml(section_manual)
assert 'disabled="disabled"' not in html_manual  # slider enabled
assert 'value="5"' in html_manual  # current count reflected
print("Card count section manual mode tests passed!")

## Trigger Button

In [ ]:
#| export
def render_settings_trigger(
    modal_id: str,                                # ID of the settings modal dialog to open
    icon_size: IconSize = icons.ghost_button,     # lucide icon size (V11.R3 ghost-button: "full" — pairs with V1.modal_disclosure at btn-xs)
) -> Button:                                      # ghost button with sliders-horizontal icon
    """Render a settings icon button that opens the card stack settings modal."""
    return Button(
        lucide_icon("sliders-horizontal", size=icon_size),
        cls=combine_classes(buttons.modal_disclosure, btn_modifiers.circle),
        title="Card stack settings",
        onclick=f"document.getElementById('{modal_id}').showModal();",
        type="button",
    )

In [ ]:
trigger = render_settings_trigger("test-settings-modal")
html = to_xml(trigger)
assert 'showModal' in html
assert 'test-settings-modal' in html
assert 'title="Card stack settings"' in html
assert 'svg' in html.lower()  # has icon
print("Trigger button tests passed!")

## Full Modal Component

In [ ]:
#| export
def render_card_stack_settings_modal(
    config: CardStackConfig,       # card stack configuration
    ids: CardStackHtmlIds,         # HTML IDs for this instance
    current_count: int = 1,        # currently selected card count
    is_auto_mode: bool = True,     # whether auto-adjust mode is active
    card_width: int = 80,          # current card width in rem
    show_scale: bool = False,      # whether to include the scale slider
    card_scale: int = 100,         # current scale percentage (if show_scale=True)
    title: str = "Display Settings",  # modal title text
) -> tuple[FT, FT]:               # (modal_dialog, trigger_button)
    """Render a modal-based card stack settings panel.

    Returns two components:
    - `modal_dialog`: The Dialog element (place anywhere in page)
    - `trigger_button`: Small settings icon button (place in toolbar)
    """
    prefix = config.prefix

    # --- Build sections ---
    sections = []

    # Card count section
    sections.append(_render_card_count_section(
        config, ids, current_count=current_count, is_auto_mode=is_auto_mode,
    ))

    # Width section
    width_oninput = f"window.cardStacks['{prefix}'].updateWidth(this.value)"
    sections.append(Div(
        _render_section_label("Width"),
        _render_slider_with_labels(
            slider_id=ids.width_slider,
            min_val=config.card_width_min,
            max_val=config.card_width_max,
            step_val=config.card_width_step,
            current_val=card_width,
            low_label="Narrow",
            high_label="Wide",
            oninput=width_oninput,
        ),
        cls=w.full,
    ))

    # Scale section (opt-in)
    if show_scale:
        scale_oninput = f"window.cardStacks['{prefix}'].updateScale(this.value)"
        sections.append(Div(
            _render_section_label("Scale"),
            _render_slider_with_labels(
                slider_id=ids.scale_slider,
                min_val=config.card_scale_min,
                max_val=config.card_scale_max,
                step_val=config.card_scale_step,
                current_val=card_scale,
                low_label="Smaller",
                high_label="Larger",
                oninput=scale_oninput,
            ),
            cls=w.full,
        ))

    # --- Build modal ---
    modal_dialog = Dialog(
        Div(
            # Close button (top-right corner)
            Form(
                Button(
                    "✕",
                    cls=combine_classes(
                        buttons.soft_dismissal, btn_modifiers.circle,
                        position.absolute, right._2, top._2,
                    ),
                ),
                method="dialog",
            ),
            # Title
            H3(
                lucide_icon("sliders-horizontal", size=icons.section_header, cls=str(m.r(2))),
                title,
                cls=combine_classes(
                    font_size.lg, font_weight.bold,
                    flex_display, items.center,
                ),
            ),
            # Settings sections
            Div(
                *sections,
                cls=combine_classes(
                    flex_display, flex_direction.col, gap(5), p.t(4),
                ),
            ),
            cls=combine_classes(modal_box, max_w.sm),
        ),
        # Backdrop (click outside to close)
        Form(Button("close"), method="dialog", cls=str(modal_backdrop)),
        id=ids.settings_modal,
        cls=str(modal),
    )

    trigger = render_settings_trigger(modal_id=ids.settings_modal)

    return modal_dialog, trigger

In [ ]:
# Test full modal — default (auto mode, no scale)
_reset_prefix_counter()
config = CardStackConfig(prefix="test")
ids = CardStackHtmlIds(prefix="test")

modal_dialog, trigger = render_card_stack_settings_modal(config, ids)

modal_html = to_xml(modal_dialog)
trigger_html = to_xml(trigger)

# Modal structure
assert 'id="test-settings-modal"' in modal_html
assert 'modal-box' in modal_html
assert 'modal-backdrop' in modal_html
assert 'Display Settings' in modal_html
assert '\u2715' in modal_html  # close button

# Card count section present
assert 'Cards' in modal_html
assert 'Auto' in modal_html
assert 'test-card-count-slider' in modal_html
assert 'test-card-count-auto-toggle' in modal_html

# Width section present
assert 'Width' in modal_html
assert 'Narrow' in modal_html
assert 'Wide' in modal_html
assert 'test-width-slider' in modal_html

# Scale section NOT present by default
assert 'Scale' not in modal_html
assert 'Smaller' not in modal_html
assert 'test-scale-slider' not in modal_html

# Trigger
assert 'showModal' in trigger_html
assert 'test-settings-modal' in trigger_html

print("Full modal default tests passed!")

In [ ]:
# Test with scale enabled
modal_with_scale, _ = render_card_stack_settings_modal(
    config, ids, show_scale=True, card_scale=150,
)
html_scale = to_xml(modal_with_scale)
assert 'Scale' in html_scale
assert 'Smaller' in html_scale
assert 'Larger' in html_scale
assert 'test-scale-slider' in html_scale
assert 'value="150"' in html_scale
print("Scale opt-in tests passed!")

# Test manual mode
modal_manual, _ = render_card_stack_settings_modal(
    config, ids, current_count=5, is_auto_mode=False, card_width=60,
)
html_manual = to_xml(modal_manual)
assert 'value="60"' in html_manual  # width value
assert 'value="5"' in html_manual  # card count value
print("Manual mode tests passed!")

In [ ]:
# Test multi-instance uniqueness
config_a = CardStackConfig(prefix="seg")
config_b = CardStackConfig(prefix="align")
ids_a = CardStackHtmlIds(prefix="seg")
ids_b = CardStackHtmlIds(prefix="align")

modal_a, trigger_a = render_card_stack_settings_modal(config_a, ids_a)
modal_b, trigger_b = render_card_stack_settings_modal(config_b, ids_b)

html_a = to_xml(modal_a)
html_b = to_xml(modal_b)

assert 'seg-settings-modal' in html_a
assert 'align-settings-modal' in html_b
assert 'seg-card-count-slider' in html_a
assert 'align-card-count-slider' in html_b
assert 'seg-width-slider' in html_a
assert 'align-width-slider' in html_b

# Triggers point to correct modals
assert 'seg-settings-modal' in to_xml(trigger_a)
assert 'align-settings-modal' in to_xml(trigger_b)

print("Multi-instance uniqueness tests passed!")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()